# Avaliação Modelo 2 ( corretor )

In [ ]:
import re
import os
import string
import unicodedata
import pandas as pd
import spacy as scy
from joblib import load
from itertools import product
from unidecode import unidecode
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

In [ ]:
PATH_BASE_TESTES_FINAIS = os.getenv("PATH_BASE_TESTES_FINAIS")
PATH_GABARITO_CORRETOR = os.getenv("PATH_GABARITO_CORRETOR")
PATH_DADOS_ANONIMIZACAO_CORRETOR = os.getenv("PATH_DADOS_ANONIMIZACAO_CORRETOR")

## Passo 1) Carregar o modelo1

O modelo1 inicial será utilizado para realização da predição dos tokens dos relatos. A saída desse modelo será utilizada para obtenção dos dados de entrada para o modelo2.

In [ ]:
rf_e2 = load('../../rf_exp2.joblib')

## Passo 2) Carregar o modelo Corretor

In [ ]:
rf_corretor = load('../../rf_corretor.joblib')

## Passo 3) Carregar os dados para predição

In [ ]:
base_relato = pd.read_csv(PATH_BASE_TESTES_FINAIS)

In [ ]:
base_relato.shape

### EDA

In [ ]:
gab = pd.read_csv(PATH_GABARITO_CORRETOR)
gab.shape

In [ ]:
teste_base_relato = base_relato.copy()

In [ ]:
lista_nomes = []
lista_palavras = []

In [ ]:
lista_nomes = []
lista_palavras = []

for n_bop, text in zip(teste_base_relato['n_bop'], teste_base_relato['relato']): 
    palavras = text.split()
    df = pd.DataFrame(palavras, columns=['palavra'])

    for _, row in df.iterrows():
        y_true_row = gab[gab['palavra'] == row['palavra']]

        if not y_true_row.empty:
            y_true_value = y_true_row['y'].iloc[0]

            if y_true_value == 0: 
                lista_palavras.append({'palavra': row['palavra'], 'y_palavra': y_true_value})
    
            elif y_true_value == 1:
                lista_nomes.append({'nome': row['palavra'], 'y_nome': y_true_value})

In [ ]:
nomes = pd.DataFrame(lista_nomes)
nomes

In [ ]:
nomes.drop_duplicates()

In [ ]:
palavras = pd.DataFrame(lista_palavras)
palavras

In [ ]:
palavras.drop_duplicates()

## Passo 4) Criar lista de saída

### Passo 4.1) Pré-processamento dos relatos

Aqui são realizadas duas etapas a primeira o pré-processamento dos relatos, onde vamos adequar os relatos para ser a entrada do modelo1 removendo stopwors, caracteres especiais e etc...

### Passo 4.2) Realização da predição dos dados com o modelo1

A partir dos dados pré-processados, entramos na etapa de predição utilizando o modelo1.

Após a predição é retornado um dafaframe contendo os dados rotulados. Para cada relato é inserido um caracter coringa que será explicado logo a frente.

In [ ]:
######################################################################################################################################################
# Pré-Processamento dos relatos
######################################################################################################################################################
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        # Remoção de números e pontuação
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

print("Iniciando pré-processamento...")
base_relato['relato'] = base_relato['relato'].apply(normalize_text)
base_relato['relato_tratado'] = base_relato['relato'].apply(preprocess)
print("Pré-processamento finalizado!")
######################################################################################################################################################
# Predição utilizando o modelo1
######################################################################################################################################################
def loadOutputList(text):
    rf_exp_2 = load('../../rf_exp2.joblib')

    output_list = []

    text = text.split()
    df = pd.DataFrame(text, columns=['palavra'])

    df['y_palavra'] = rf_exp_2.predict(df['palavra'].to_list())

    for _, row in df.iterrows():
        output_list.append({
            'palavra': row['palavra'],
            'y_palavra': int(row['y_palavra']),
        })
    output_list.append({
            'palavra': '#',
            'y_palavra': 2,
        })
    
    output_list.pop()

    return pd.DataFrame(output_list)
# print("Carregando predição do modelo...")
# output_list = loadOutputList(base_relato['relato_tratado'][0])
# print("Base de dados de predição criada!")
######################################################################################################################################################

essa é a estrutura gerada após a predição do modelo1

In [ ]:
# output_list

### Estratégia utilizada para separação dos textos no dataframe abaixo:

É evidente que os relatos não estão dispostos em texto corrido.  

Qual o passo a passo?  

Passo 1) Pré-processamento: Aqui vamos remover stopwords, caracteres especiais e etc 
Logo, após isso vamos realizar a tokenização do relato, transformando o memso em uma vetor de tokens que será armazenada em um dataframe.  

Passo 2) Inserção do delimitador utilizando caracter coringa '#' : Após a inserção de cada vetor de tokens um caractere coringa será inserido ao final do processo de inserção. Este por sua vez tem o papel de delimitar o inicio e fim de um documento de texto tokenizado.

Após a criação da lista de saída (output_list) é realizada a validação da quantidade de '#', verificando se a mesma corresponde a quantidade de relatos na base

In [ ]:
# output_list[output_list['palavra'] == '#']

## Passo 5) Criar lista temporária

- Para cada item, realizar a predição com o modelo do João
- Inserir a predição na lista de saída.
- Se a predição for negativa, inserir a predição na lista temporária, guardando o indice na lista de saída.

In [ ]:
def vector_input_corretor(output_list : pd.DataFrame):
    
    def LoadPedSucWord(output_list : pd.DataFrame):
        list_aux_psw = []
        df_aux = output_list.copy()
        # print(df_aux)

        df_aux['predecessor'] = df_aux['palavra'].shift(1).fillna('#')
        df_aux['y_predecessor'] = df_aux['predecessor'].apply(
            lambda x: df_aux[df_aux['palavra'] == x]['y_palavra'].iloc[0] if x != '#' else 2
        )

        df_aux['sucessor'] = df_aux['palavra'].shift(-1).fillna('#')
        df_aux['y_sucessor'] = df_aux['sucessor'].apply(
            lambda x: df_aux[df_aux['palavra'] == x]['y_palavra'].iloc[0] if x != '#' else 2
        )

        # print(df_aux)

        for _, row in df_aux.iterrows():
            list_aux_psw.append({
                'predecessor': row['predecessor'],
                'y_predecessor': int(row['y_predecessor']),
                'palavra': row['palavra'],
                'y_palavra': int(row['y_palavra']),
                'sucessor': row['sucessor'],
                'y_sucessor': int(row['y_sucessor']),
            })

        return pd.DataFrame(df_aux)

    print("Realizando coleta de predecessores e sucessores...")
    temp_list = LoadPedSucWord(output_list)
    temp_list = temp_list[temp_list['y_palavra'] == 0].reset_index()
    indices_originais = temp_list['index']
    print("Predecessores e Sucessores coletados!")

    def bigrams_vector(lista_tokens):
        vectorizer = load('../../vectorizer.joblib')
        alfabeto = list(string.ascii_uppercase)
        alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]
        vector = vectorizer.transform(lista_tokens)
        return pd.DataFrame(vector.todense(), columns=alfabeto_bigrams)

    print("Criando vetor de bigramas para os tokens...")
    token = temp_list['palavra']
    tokens_vector_bigrams = bigrams_vector(token.to_list())
    print("Vetores de bigramas dos tokens criados!")
    print("Concatenando vetor de bigramas com dados de predecessores e sucessores dos tokens...")
    tokens_vector_bigrams = pd.concat([tokens_vector_bigrams, temp_list[['predecessor','y_predecessor', 'palavra', 'y_palavra','sucessor', 'y_sucessor']]], axis=1)
    print("Concatenação concluída!")

    def create_base_cg(lista_tokens):
        nlp = scy.load('pt_core_news_lg')
        def renomear(doc):
            mapeamento_pos = {
                'NOUN': '<SUBSTANTIVO>',
                'PROPN': '<SUBSTANTIVO>',
                'VERB': '<VERBO>',
                'AUX': '<VERBO>',
                'ADJ': '<ADJETIVO>',
                'ADV': '<ADVERBIO>',
                'PUNCT': '<PONTUACAO>',
                'CCONJ': '<CONJUNCAO>',
                'SCONJ': '<CONJUNCAO>',
                'PRON': '<PRONOME>',
                'DET': '<ARTIGO>',
                'ADP': '<PREPOSIÇÃO>',
                'NUM': '<NUMERAL>',
                'SYM': '<SIMBOLO>',
                'X': '<PALAVRA ESTRANGEIRA>',
                'SPACE': '<ESPACO>',
            }
            
            return [mapeamento_pos.get(token.pos_, token.pos_) for token in doc]


        word_class = []
        for token in lista_tokens:
            if token == '#':
                word_class.append('<NULL>')
            else:
                lreview = token.lower()
                ldoc = nlp(lreview)
                classes = renomear(ldoc)
                for i in range(len(token.split())):
                    word_class.append(classes[i])

        return pd.DataFrame({'palavra':lista_tokens, 'classe':word_class})

    print("Realizando classificação gramatical dos predecessores, sucessores e dos tokens...")
    tokens_vector_bigrams['cg_p'] = create_base_cg(tokens_vector_bigrams['predecessor'])['classe']
    tokens_vector_bigrams['cg_w'] = create_base_cg(tokens_vector_bigrams['palavra'])['classe']
    tokens_vector_bigrams['cg_s'] = create_base_cg(tokens_vector_bigrams['sucessor'])['classe']
    print("Classificação gramatical realizada com sucesso!")
    def transformLabelsCG(tokens_vector_bigrams : pd.DataFrame):
        lb = LabelEncoder()

        labels = ['<SUBSTANTIVO>', '<VERBO>', '<ADJETIVO>', '<ADVERBIO>', '<PONTUACAO>',
                '<CONJUNCAO>', '<PRONOME>', '<ARTIGO>', '<PREPOSIÇÃO>', '<NUMERAL>',
                '<SIMBOLO>', '<PALAVRA ESTRANGEIRA>', '<ESPACO>', '<NULL>']

        lb.fit(labels)
        
        tokens_vector_bigrams['cg_p'] = lb.transform(tokens_vector_bigrams['cg_p'])
        tokens_vector_bigrams['cg_w'] = lb.transform(tokens_vector_bigrams['cg_w'])
        tokens_vector_bigrams['cg_s'] = lb.transform(tokens_vector_bigrams['cg_s'])

        return tokens_vector_bigrams
    print("Realizando a transformação dos labels da classificação gramatical para entrada adequada do modelo...")
    tokens_vector_bigrams = transformLabelsCG(tokens_vector_bigrams)
    print("Transformação dos labels da classificação gramatical realizada com sucesso!")
    print("Removendo as colunas contendo os tokens dos predecessores, sucessores e palavras...")
    final_tokens_vector_bigrams = tokens_vector_bigrams.drop(['sucessor', 'predecessor','palavra','y_palavra'], axis=1)
    print("Remoção realizada com sucesso!")
    print("Retornando vetor de entrada para o modelo.")

    return final_tokens_vector_bigrams, indices_originais

In [ ]:
# vector_input_temp_list, indices_originais = vector_input_corretor(output_list)

In [ ]:
# vector_input_temp_list

In [ ]:
def predict_corretor(vector_input_temp_list, indices_originais, output_list):
    result = rf_corretor.predict(vector_input_temp_list)
    temp_list = pd.DataFrame({'y_pred':result})
    temp_list = pd.concat([temp_list, indices_originais], axis=1)
    select_index = output_list['y_palavra'][temp_list['index'].to_list()]
    temp_list = temp_list.dropna(subset=['y_pred']).astype(int)

    for idx, pred_Ex2, pred_Corretor in zip(temp_list['index'].to_list(), select_index.to_list(), temp_list['y_pred'].to_list()):
        if int(pred_Corretor) == 1:
            output_list['y_palavra'][idx] = pred_Corretor
            palavra = output_list['palavra'][idx]
    
    return output_list

In [ ]:
base_relato_teste = base_relato.copy()

In [ ]:
base_relato_teste

In [ ]:
def check_names(output_list : pd.DataFrame):
    gab = pd.read_csv(PATH_GABARITO_CORRETOR)
    copia_output_list = output_list.copy()
    
    copia_output_list = copia_output_list[copia_output_list['y_palavra'] != 2]
    copia_output_list['y_palavra'] = copia_output_list['y_palavra'].astype(int)

    merge_gab_output_list = copia_output_list.merge(gab[['palavra', 'y']], left_on='palavra', right_on='palavra', how='left')
    merge_gab_output_list['y'] = merge_gab_output_list['y'].fillna(0).astype(int)
    merge_gab_output_list = merge_gab_output_list[merge_gab_output_list['y_palavra'] != 2]

    return merge_gab_output_list

In [ ]:
for idx, text in base_relato_teste['relato_tratado'].items():
    output_list = loadOutputList(text)
    vector_input_temp_list, indices_originais = vector_input_corretor(output_list)
    output_list = predict_corretor(vector_input_temp_list, indices_originais, output_list)
    pred_gab_names = check_names(output_list)
    base_relato_teste.at[idx, 'corretor_qtd_tokens'] = len(output_list)
    base_relato_teste.at[idx, 'corretor_qtd_pred_nomes'] = len(pred_gab_names[pred_gab_names['y_palavra'] == 1])
    base_relato_teste.at[idx, 'corretor_qtd_gab_nomes'] = len(pred_gab_names[pred_gab_names['y'] == 1])
    base_relato_teste.at[idx, 'corretor_nomes_acertados'] = len(pred_gab_names[(pred_gab_names['y'] == 1) & (pred_gab_names['y_palavra'] == 1)])

In [ ]:
1+1 

In [ ]:
print(2+2)

In [ ]:
base_relato_teste.to_csv(PATH_DADOS_ANONIMIZACAO_CORRETOR, index=False)